# Day 17 — Advanced Anomaly / Forecast Model

**Purpose:** train and validate a proportionate candidate forecasting model and save it for later inference. This notebook compares Random Forest and Gradient Boosting using a chronological 70/15/15 split.

**Important limitation:** the dataset's timestamps were synthetically generated from a static water-potability table. Therefore, results are exploratory pipeline tests and must not be presented as evidence of real underwater temporal forecasting or real sensor anomalies.

## 1. Setup and load data
Run this notebook in the same folder as `water_potability_temporal_synthetic.csv`. In Colab, upload the CSV first or mount Drive and update `DATA_PATH`.

In [ ]:
import os, json, joblib
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

DATA_PATH = "water_potability_temporal_synthetic.csv"
df = pd.read_csv(DATA_PATH)
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
print("Raw shape:", df.shape)
print("Rows with missing timestamps:", df["timestamp"].isna().sum())
print("Duplicate timestamp rows:", df["timestamp"].duplicated().sum())

## 2. Timestamp handling and feature definitions
Rows with missing timestamps are excluded from this time-ordered experiment. For duplicate timestamps, keep the first row in stable input order. No hourly resampling or interpolation is applied because the timestamps are synthetic.

Features use only information available at the current observation: nine current sensor measurements, hour/day/month, previous pH, trailing three-observation pH mean and standard deviation, and elapsed time since the previous timestamp. The target is the **next row's pH**. Rolling statistics are trailing (not centered), so they do not look ahead.

In [ ]:
df = df.dropna(subset=["timestamp"]).sort_values("timestamp", kind="stable")
df = df.drop_duplicates(subset=["timestamp"], keep="first").reset_index(drop=True)
sensor_cols = ["ph", "Hardness", "Solids", "Chloramines", "Sulfate",
               "Conductivity", "Organic_carbon", "Trihalomethanes", "Turbidity"]
features = pd.DataFrame(index=df.index)
for col in sensor_cols:
    features[col + "_current"] = df[col]
features["hour"] = df.timestamp.dt.hour
features["day_of_week"] = df.timestamp.dt.dayofweek
features["month"] = df.timestamp.dt.month
features["ph_lag_1"] = df.ph.shift(1)
features["ph_rolling_mean_3"] = df.ph.rolling(3, min_periods=1).mean()
features["ph_rolling_std_3"] = df.ph.rolling(3, min_periods=2).std()
features["hours_since_previous"] = df.timestamp.diff().dt.total_seconds() / 3600
data = features.copy()
data["target"] = df.ph.shift(-1)
data["timestamp"] = df.timestamp
data = data.dropna(subset=["target"]).reset_index(drop=True)
X = data.drop(columns=["target", "timestamp"])
y = data["target"]
print("Model rows:", len(data), "| Feature count:", X.shape[1])
print("Features:", list(X.columns))

## 3. Chronological split and leakage controls
Use earliest 70% for training, next 15% for validation, and latest 15% for final test. The median imputer is inside each model pipeline, so medians are learned from training data only. Model selection uses validation RMSE; test data is reserved for the final report.

In [ ]:
n = len(data)
n_train = int(n * 0.70)
n_val = int(n * 0.15)
X_train, X_val, X_test = X.iloc[:n_train], X.iloc[n_train:n_train+n_val], X.iloc[n_train+n_val:]
y_train, y_val, y_test = y.iloc[:n_train], y.iloc[n_train:n_train+n_val], y.iloc[n_train+n_val:]
print({"train": len(X_train), "validation": len(X_val), "test": len(X_test)})
print("Chronological boundaries:", data.timestamp.iloc[0], data.timestamp.iloc[n_train-1], data.timestamp.iloc[n_train], data.timestamp.iloc[-1])

## 4. Train comparable candidate models
A Random Forest is used as a tree-ensemble baseline; Gradient Boosting is the stronger candidate. These are regression models for next-observation pH, not anomaly detectors. Complexity is kept moderate for this small tabular dataset.

In [ ]:
models = {
    "RandomForestRegressor": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestRegressor(n_estimators=250, min_samples_leaf=3,
                                         max_features=0.8, random_state=42, n_jobs=-1))
    ]),
    "GradientBoostingRegressor": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", GradientBoostingRegressor(n_estimators=150, learning_rate=0.04,
                                             max_depth=2, loss="huber", random_state=42))
    ])
}
results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    val_pred = model.predict(X_val)
    test_pred = model.predict(X_test)
    results.append({
        "model": name,
        "validation_MAE": mean_absolute_error(y_val, val_pred),
        "validation_RMSE": mean_squared_error(y_val, val_pred) ** 0.5,
        "test_MAE": mean_absolute_error(y_test, test_pred),
        "test_RMSE": mean_squared_error(y_test, test_pred) ** 0.5
    })
metrics = pd.DataFrame(results).sort_values("validation_RMSE")
display(metrics)

## 5. Select and save candidate model
Choose the model with the lowest validation RMSE. Save the complete preprocessing + estimator pipeline, feature names, target definition, and split counts. The saved file can be loaded with `joblib.load`.

In [ ]:
best_name = metrics.iloc[0]["model"]
best_model = models[best_name]
model_file = "candidate_forecast_model.joblib"
joblib.dump(best_model, model_file)
metadata = {
    "model": best_name,
    "target": "next_row_ph",
    "features": list(X.columns),
    "feature_count": int(X.shape[1]),
    "selection_rule": "Lowest validation RMSE",
    "training_rows": len(X_train), "validation_rows": len(X_val), "test_rows": len(X_test),
    "timestamp_note": "Synthetic timestamps; exploratory pipeline validation only.",
    "preprocessing": "Median imputation fitted inside training-only pipeline."
}
with open("candidate_model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)
metrics.to_csv("day17_advanced_model_metrics.csv", index=False)
print("Saved:", model_file, "and candidate_model_metadata.json")

## 6. Final holdout predictions
Generate predictions on the untouched chronological test partition and save them for plotting/reporting. A high error or unstable metric should be discussed rather than hidden.

In [ ]:
test_pred = best_model.predict(X_test)
predictions = pd.DataFrame({
    "timestamp": data.timestamp.iloc[n_train+n_val:].values,
    "actual_next_ph": y_test.values,
    "predicted_next_ph": test_pred
})
predictions.to_csv("day17_candidate_test_predictions.csv", index=False)
print("Final test MAE:", mean_absolute_error(y_test, test_pred))
print("Final test RMSE:", mean_squared_error(y_test, test_pred) ** 0.5)
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(10, 4))
    plt.plot(predictions.timestamp, predictions.actual_next_ph, label="Actual", linewidth=1)
    plt.plot(predictions.timestamp, predictions.predicted_next_ph, label="Predicted", linewidth=1)
    plt.title("Day 17 — Actual vs Predicted next-row pH (synthetic timestamps)")
    plt.xlabel("Synthetic timestamp"); plt.ylabel("pH"); plt.legend(); plt.tight_layout(); plt.show()
except ImportError:
    print("Install matplotlib to display the plot.")

## 7. Interpretation and limitations
- Compare models using validation RMSE for selection and test MAE/RMSE for final holdout reporting.
- Next-row pH is a proxy target chosen to demonstrate forecasting workflow; it is not a verified operational forecast target.
- The source table is cross-sectional water-potability data. Synthetic timestamps do not create genuine temporal dependence.
- Do not claim real underwater sensor forecasting performance until real timestamped sensor data is collected and the experiment is repeated.
- No advanced sequence model is justified at this stage; a modest tree ensemble is proportionate to the available data.